In [1]:
import pyarrow.parquet as pq
import pyarrow as pa
path_pq = r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\periods_data\1950_1999\base_data\shard_00386.parquet"
path_pq_new = r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\periods_data\1950_1999\base_data\shard_00386_fixed.parquet"
table = pq.read_table(path_pq)
num_rows = len(table)
# Write with enough row groups (at least 8, ideally more)
row_group_size = max(1, num_rows // 16)  # 16 row groups for safety
pq.write_table(table, path_pq_new, row_group_size=row_group_size)


In [3]:
table2 = pq.read_table(path_pq_new)

In [2]:
table

pyarrow.Table
text: string
----
text: [["Human LIG-1 homolog (HLIG-1)

ABSTRACT

HLIG-1 polypeptides and polynucleotides and methods for pr (... 49529 chars omitted)","Explosive gunnery target apparatus

ABSTRACT

An apparatus for providing immediate and dramatic in (... 37820 chars omitted)","SAND, Justice.
Adeline Johnson (Adeline) appealed from an order appointing Herman W. Bischof (Bisc (... 11732 chars omitted)","PER CURIAM.
This is an interlocutory appeal from an order of the circuit court denying the motion  (... 7613 chars omitted)","Method for the secondary cooling of a metal strand

ABSTRACT

Method and apparatus for secondary c (... 45151 chars omitted)",...,"Method of and arrangement for applying a fluid to a surface using a vibrating needle

ABSTRACT

Th (... 11051 chars omitted)","Printing apparatus and printing method with security protection for confidential data

ABSTRACT

T (... 41639 chars omitted)","O'CONNELL, C.J.
This is an action brought by a real estate salesman

In [4]:
table2

pyarrow.Table
text: string
----
text: [["Human LIG-1 homolog (HLIG-1)

ABSTRACT

HLIG-1 polypeptides and polynucleotides and methods for pr (... 49529 chars omitted)","Explosive gunnery target apparatus

ABSTRACT

An apparatus for providing immediate and dramatic in (... 37820 chars omitted)","SAND, Justice.
Adeline Johnson (Adeline) appealed from an order appointing Herman W. Bischof (Bisc (... 11732 chars omitted)","PER CURIAM.
This is an interlocutory appeal from an order of the circuit court denying the motion  (... 7613 chars omitted)","Method for the secondary cooling of a metal strand

ABSTRACT

Method and apparatus for secondary c (... 45151 chars omitted)",...,"5. A multilayer headbox as defined in claim 3 or claim 4, together with means for supplying gaseou (... 7063 chars omitted)","Toroidal low-field reactive gas source

ABSTRACT

An apparatus for dissociating gases includes a p (... 45109 chars omitted)","GRIFFIN, P. J.
Petitioner seeks a writ of prohibition restraining th

In [1]:
import pyarrow.parquet as pq
import os

In [3]:
pwd

'C:\\Users\\danielyoon\\Dropbox\\hist_LLM'

In [7]:
base = os.path.expanduser(r'C:\Users\danielyoon\Dropbox\hist_LLM\training_data\1950_1999\base_data')
f = sorted([x for x in os.listdir(base) if x.endswith('.parquet')])[0]
pf = pq.ParquetFile(os.path.join(base, f))
print(f'File: {f}')
print(f'Row groups: {pf.num_row_groups}')
print(f'Total rows: {pf.metadata.num_rows}')

File: shard_00000.parquet
Row groups: 11
Total rows: 11248


In [ ]:
python -c "
import pyarrow.parquet as pq
import pandas as pd
import os
base = os.path.expanduser('~/hist-llm-data-period/training_data/1950_1999/base_data')
path = os.path.join(base, 'shard_00386.parquet')
df = pd.read_parquet(path)
df.to_parquet(path, row_group_size=681)
pf = pq.ParquetFile(path)
print(f'New row groups: {pf.num_row_groups}')
print(f'Total rows: {pf.metadata.num_rows}')
"

In [ ]:
python -c "
import pyarrow.parquet as pq
import os
base = os.path.expanduser('~/hist-llm-data-period/training_data/1950_1999/base_data')
files = sorted([x for x in os.listdir(base) if x.endswith('.parquet')])
print(f'Total files: {len(files)}')
print(f'First file: {files[0]}')
print(f'Last file: {files[-1]}')
pf_last = pq.ParquetFile(os.path.join(base, files[-1]))
print(f'Last file row groups: {pf_last.num_row_groups}')
print(f'Last file total rows: {pf_last.metadata.num_rows}')
pf_second_last = pq.ParquetFile(os.path.join(base, files[-2]))
print(f'Second-to-last file row groups: {pf_second_last.num_row_groups}')
print(f'Second-to-last file total rows: {pf_second_last.metadata.num_rows}')
"

In [ ]:
python -c "
import pyarrow.parquet as pq
import os
base = os.path.expanduser('~/hist-llm-data-period/training_data/1950_1999/base_data')
files = sorted([x for x in os.listdir(base) if x.endswith('.parquet')])
for f in files:
  pf = pq.ParquetFile(os.path.join(base, f))
  if pf.num_row_groups < 8:
      print(f'{f}: {pf.num_row_groups} row groups, {pf.metadata.num_rows} rows')
"

Then fix all of them at once:

python -c "
import pyarrow.parquet as pq
import pandas as pd
import os
base = os.path.expanduser('~/hist-llm-data-period/training_data/1950_1999/base_data')
files = sorted([x for x in os.listdir(base) if x.endswith('.parquet')])
for f in files:
  path = os.path.join(base, f)
  pf = pq.ParquetFile(path)
  if pf.num_row_groups < 8:
      rows = pf.metadata.num_rows
      new_rg_size = max(1, rows // 8)
      df = pd.read_parquet(path)
      df.to_parquet(path, row_group_size=new_rg_size)
      pf2 = pq.ParquetFile(path)
      print(f'Fixed {f}: {pf.num_row_groups} -> {pf2.num_row_groups} row groups')
"

In [ ]:
python -c "
import pyarrow.parquet as pq
import os
base = os.path.expanduser('~/hist-llm-data-period/training_data/1950_1999/base_data')
f = sorted([x for x in os.listdir(base) if x.endswith('.parquet')])[0]
pf = pq.ParquetFile(os.path.join(base, f))
print(f'File: {f}')
print(f'Row groups: {pf.num_row_groups}')
print(f'Total rows: {pf.metadata.num_rows}')
"